In [1]:
# make sure you ran workspace\ibm-qiskit-login.py first


print("let's try qiskit")
# https://docs.quantum.ibm.com/guides/hello-world

let's try qiskit


In [2]:
import qiskit_ibm_runtime
qiskit_ibm_runtime.version.get_version_info()

'0.38.0'

In [3]:
import qiskit
qiskit.version.get_version_info()

'2.0.0'

In [4]:
# new!  test boolean op circuits
# Part A - set up 

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit import QuantumCircuit, transpile #, assemble
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator, SamplerV2 as Sampler

# Read default credentials from disk
service = QiskitRuntimeService(channel = 'ibm_quantum')

qc = QuantumCircuit(6)
#qc.x(0)
qc.x(1)
#qc.x(2)
#qc.x(3)

qc.cx(0, 4)
qc.cx(1, 4)
qc.cx(2, 5)
qc.cx(3, 5)


qc.barrier()
qc.cx(4,5)

#qc.mcx([0,1],2)

qc.measure_all()
print(qc.draw())

C:\Users\thromadka\AppData\Local\Temp\ipykernel_27952\1507800428.py:10: DeprecationWarning: The "ibm_quantum" channel option is deprecated and will be sunset on 1 July. After this date, ibm_cloud will be the only valid channel. For information on migrating to the new IBM Quantum Platform on the "ibm_cloud" channel, review the migration guide https://quantum.cloud.ibm.com/docs/migration-guides/classic-iqp-to-cloud-iqp .
  service = QiskitRuntimeService(channel = 'ibm_quantum')


                             ░       ░ ┌─┐               
   q_0: ───────■─────────────░───────░─┤M├───────────────
        ┌───┐  │             ░       ░ └╥┘┌─┐            
   q_1: ┤ X ├──┼────■────────░───────░──╫─┤M├────────────
        └───┘  │    │        ░       ░  ║ └╥┘┌─┐         
   q_2: ──■────┼────┼────────░───────░──╫──╫─┤M├─────────
          │    │    │        ░       ░  ║  ║ └╥┘┌─┐      
   q_3: ──┼────┼────┼────■───░───────░──╫──╫──╫─┤M├──────
          │  ┌─┴─┐┌─┴─┐  │   ░       ░  ║  ║  ║ └╥┘┌─┐   
   q_4: ──┼──┤ X ├┤ X ├──┼───░───■───░──╫──╫──╫──╫─┤M├───
        ┌─┴─┐└───┘└───┘┌─┴─┐ ░ ┌─┴─┐ ░  ║  ║  ║  ║ └╥┘┌─┐
   q_5: ┤ X ├──────────┤ X ├─░─┤ X ├─░──╫──╫──╫──╫──╫─┤M├
        └───┘          └───┘ ░ └───┘ ░  ║  ║  ║  ║  ║ └╥┘
meas: 6/════════════════════════════════╩══╩══╩══╩══╩══╩═
                                        0  1  2  3  4  5 


In [5]:
# Part B - run job
from qiskit.visualization import plot_histogram

# Optimize problem for quantum execution
backend = service.least_busy(operational=True, simulator=False)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = Sampler(mode=backend)
sampler.options.default_shots = 5000  #1000 is enough for the simple tests, #10000 for 4-qb MCX
job = sampler.run([isa_circuit])
print(f"Job ID is {job.job_id()}")
pub_result = job.result()[0]
print(f"Counts for the meas output register: {pub_result.data.meas.get_counts()}")


plot_histogram(pub_result.data.meas.get_counts())

Job ID is d0wbt2ghtw7g008py0ag


RequestsApiError: 'HTTPSConnectionPool(host=\'api.quantum.ibm.com\', port=443): Max retries exceeded with url: /runtime/jobs/d0wbt2ghtw7g008py0ag?exclude_params=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000012606D965F0>: Failed to resolve \'api.quantum.ibm.com\' ([Errno 11001] getaddrinfo failed)"))'